# EcoSort AI: build the dataset and train the detector

Run the cells from top to bottom in Google Colab.

Before you start: **Runtime → Change runtime type → T4 GPU**.

## 1. Get the code and libraries

In [ ]:
!git clone https://github.com/TheAnakin01/ecosort-ai
%cd ecosort-ai/ai-model
!pip install -q ultralytics roboflow

## 2. Download the three Roboflow datasets

Run the cell and paste your Roboflow API key when it asks (Roboflow → Settings → API Keys).
The key is not saved in the notebook.

The cell downloads the newest version of each dataset:

| Folder | Dataset |
|---|---|
| `trash-detection` | [Trash Detection](https://universe.roboflow.com/amongus-o6ilp/trash-detection-1fjjc-ipij8) |
| `trashnet` | [TrashNet: annotated images of trash](https://universe.roboflow.com/polygence-project/trashnet-a-set-of-annotated-images-of-trash-that-can-be-used-for-object-detection) |
| `neat-now` | [Neat Now](https://universe.roboflow.com/fyp-gojku/neat-now) |

In [ ]:
from getpass import getpass

from roboflow import Roboflow

# (folder name, workspace, project)
DATASETS = [
    ("trash-detection", "amongus-o6ilp", "trash-detection-1fjjc-ipij8"),
    ("trashnet", "polygence-project", "trashnet-a-set-of-annotated-images-of-trash-that-can-be-used-for-object-detection"),
    ("neat-now", "fyp-gojku", "neat-now"),
]

# .strip() removes spaces or line breaks that a phone can add when pasting.
api_key = getpass("Paste your Roboflow Private API Key: ").strip()
try:
    rf = Roboflow(api_key=api_key)
except RuntimeError:
    raise SystemExit(
        "Roboflow did not accept that key. Copy the Private API Key from "
        "app.roboflow.com -> Settings -> API Keys, then run this cell again."
    ) from None
for folder, workspace, project_name in DATASETS:
    project = rf.workspace(workspace).project(project_name)
    # A version id looks like "workspace/project/3"; take the highest number.
    latest = max(project.versions(), key=lambda v: int(str(v.id).rsplit("/", 1)[-1]))
    print(f"{folder}: downloading version {latest.id}")
    latest.download("yolov8", location=f"downloads/{folder}")

## 3. Check how the classes will be merged

This prints every class in every dataset and which of our four classes it becomes.
Nothing is copied yet.

Read the output carefully:
- A class marked `DROPPED` is left out. If it should be one of ours, add a line for it in `configs/class_overrides.yaml` and run this cell again.
- Look at **Boxes per class**. If `organic` is marked as too few, the model will be weak on organic waste until you add your own photos.

In [ ]:
!python scripts/prepare_dataset.py downloads/trash-detection downloads/trashnet downloads/neat-now --dry-run

## 4. Build the merged dataset

In [ ]:
!rm -rf datasets/ecosort
!python scripts/prepare_dataset.py downloads/trash-detection downloads/trashnet downloads/neat-now

## 5. Train

About 50 epochs is a reasonable first run. Training stops early if the model stops improving.
The last lines print mAP@50 and mAP@50-95 on the test split. Write them in `docs/training-log.md`.

In [ ]:
!python scripts/train.py --epochs 50 --name ecosort-baseline

## 6. Look at the results

In [ ]:
from IPython.display import Image, display

run = "runs/detect/ecosort-baseline"
display(Image(f"{run}/results.png", width=900))
display(Image(f"{run}/confusion_matrix.png", width=700))
display(Image(f"{run}/val_batch0_pred.jpg", width=900))

## 7. Download the trained model

Put the downloaded `best.pt` in `ai-model/weights/` on the machine that runs the backend.

In [ ]:
from google.colab import files

files.download("runs/detect/ecosort-baseline/weights/best.pt")